In [ ]:
# ECC ASAP: Sonde preparation
import os
import polars as pl
from processing.ecc import ECC
%matplotlib widget

# choose a file
file = 'tests/data/ecc_asap/20220504C37105.TXT'

# Create an instance of the ECCSONDE class
sonde = ECC()

# Use the extract_ecc_asap method to process the file
result = sonde.extract_ecc_asap(file=file)

# Plot the result
sonde.plot_ecc_asap(result, suptitle=file)

# select folder with ECC ASAP data
folder_path = '/product_data/data/pay/Kenya/NRB/OZONE_ECC_ASAP/incoming/'

for year in range(2019, 2025):
    year = str(year)
    # Compile data and metadata from the specified folder
    compiled_data = sonde.compile_ecc_asap(folder_path=os.path.join(folder_path, year))

    # Access the compiled metadata and data
    metadata_df = compiled_data['metadata']
    metadata_df.write_parquet(os.path.join('data/level1/', year, 'ecc_asap_metadata.parquet'))
    data_df = compiled_data['data']
    data_df.write_parquet(os.path.join('data/level1/', year, 'ecc_asap_data.parquet'))

    data_df = pl.read_parquet(os.path.join('data/level1/', year, 'ecc_asap_data.parquet'))

    # Plot the compiled data interactively
    path = os.path.join('results/ecc/', year)
    os.makedirs(path, exist_ok=True)

    sonde.plot_compiled_ecc_asap(data_df=data_df, suptitle=f'ECC ASAP {year}', interactive=False, path=path)

In [ ]:
from processing.ecc import SHADOZ

shadoz = SHADOZ()
target = 'data/level2/ecc/shadoz'

for year in range(2023, 2025):
    df_data, df_metadata = shadoz.download_and_extract_shadoz_zip_to_parquet(year=year, target=target)

In [ ]:
# calculate TCO from sondes, complement with residual above burst altitude from CAMS EAC4
import matplotlib.pyplot as plt
import polars as pl
import numpy as np
from processing.ecc import ECC
from toolbox.numerical_analysis_1d import interpolate_logarithmic
import datetime

ecc = ECC()

source_shadoz = 'data/level2/ecc/shadoz'
source_cams = 'data/level3/copernicus/cams/eac4'

df = pl.DataFrame()
for year in range(2020, 2023):
    try:
        # read SHADOZ data
        df_shadoz_year = pl.read_parquet(f'{source_shadoz}/ecc_sonde_data_{year}.parquet')

        # read CAMS profile data
        df_cams_year = pl.read_parquet(f'{source_cams}/cams-global-reanalysis-eac4-monthly_{year}.parquet')

        # extract launch dates
        dtm_launches = df_shadoz_year.select(pl.col('dtm')).unique().to_series().to_list()

        for dtm in dtm_launches:
            try:
                # extract data
                df_shadoz = df_shadoz_year.filter(pl.col('dtm')==dtm)

                # collect relevant metadata
                filename = df_shadoz['filename'][0]
                burst_pressure = min(df_shadoz['Press'])
                tco_shadoz = max(df_shadoz['O3_DU'])

                # re-calculate total column ozone in Dobson units
                df_shadoz = ecc.total_column_ozone_from_pressure_profile(df=df_shadoz, pressure_col='Press', ozone_col='O3_mPa')
                # tco_ecc.write_csv('data/level1/ecc/shadoz/test.csv')
                tco_shadoz_recalc = max(df_shadoz['O3_DU_calc'])

                # select corresponding CAMS profile
                df_cams = df_cams_year.filter(pl.col('dte')==datetime.date(dtm.year, dtm.month, 1))
                # print(df_cams.schema)

                # interpolate CAMS profile
                df_cams = interpolate_logarithmic(df_cams['level'], df_cams['O3_mPa'], N=5, x0=burst_pressure)
                df_cams = df_cams.rename({'x': 'Press', 'y': 'O3_mPa'}).filter(pl.col('Press') <= burst_pressure)
                df_cams = ecc.total_column_ozone_from_pressure_profile(df=df_cams, pressure_col='Press', ozone_col='O3_mPa')
                residual_tco_cams = max(df_cams['O3_DU_calc'])

                tco_shadoz_cams = tco_shadoz + residual_tco_cams
                tco_shadoz_recalc_cams = tco_shadoz_recalc + residual_tco_cams
                _df = pl.DataFrame({'filename': filename,
                                    'dte': dtm.strftime('%Y-%m-%d'),
                                    'burst_pressure_hPa': burst_pressure,
                                    'tco_shadoz': tco_shadoz,
                                    'tco_shadoz_recalc': tco_shadoz_recalc,
                                    'tco_shadoz_cams': tco_shadoz_cams,
                                    'tco_shadoz_recalc_cams': tco_shadoz_recalc_cams,
                                    })
            except Exception as err:
                print(f"year={year}: {err}")
                pass

            df = pl.concat([df, _df], how='diagonal')
        
    except Exception as err:
        print(f"year={year}: {err}")
        pass

display(df)

In [ ]:
# plot TCO from sondes
import polars as pl
df = pl.read_csv('results/ecc/tco_from_ecc.csv')

df = df.sort(by='dte')
fig = plt.figure(figsize=(10, 6))
plt.scatter(df['dte'].str.to_date(), df['tco_shadoz'], s=6, label="TCO from SHADOZ file until burst")
# plt.scatter(df['dte'].str.to_date(), df['tco_shadoz_recalc'], marker='o', c='red', s=8, label="TCO from SHADOZ file until burst (recalculated)")
plt.scatter(df['dte'].str.to_date(), df['tco_shadoz_cams'], marker='o', c='red', s=12, label="TCO from SHADOZ file until burst + residual from CAMS")
# plt.scatter(df['dte'].str.to_date(), df['tco_shadoz_recalc_cams'], marker='o', c='orange', s=8, label="TCO from SHADOZ file until burst (recalculated) + residual from CAMS")
plt.xlim(datetime.date(2000,1,1), datetime.date(2024,1,1))
plt.ylim(220, 280)
plt.xlabel('Date')
plt.ylabel('Total column ozone [DU]')
# plt.suptitle(suptitle)
plt.title('Total Column Ozone from Nairobi ECC ozone soundings')
# plt.grid(True)
plt.legend()
plt.show()
df.write_csv('results/ecc/tco_from_ecc.csv')
fig.savefig('results/ecc/tco_from_ecc.png')

In [ ]:
# plot *all* available sonde profiles, together with CAMS reanalysis data for residual above burst altitude
import datetime
import matplotlib.pyplot as plt
import polars as pl
from processing.ecc import ECC
ecc = ECC()

for year in range(2020, 2025):
    try:
        df_ecc_year = pl.read_parquet(f'data/level2/ecc/shadoz/ecc_sonde_data_{year}.parquet')

        # extract launch date
        dtm_launches = df_ecc_year.select(pl.col('dtm')).unique().to_series().to_list()

        for dtm_launch in dtm_launches:
            df_ecc = df_ecc_year.filter(pl.col('dtm')==dtm_launch)
            df_model_year = pl.read_parquet(f'data/level3/copernicus/cams/eac4/cams-global-reanalysis-eac4-monthly_{year}.parquet')
            df_model_year = df_model_year.rename({'level': 'Press'})
            df_model = df_model_year.filter(pl.col('dte')==datetime.date(dtm_launch.year, dtm_launch.month, 1))

            ecc.plot_ecc_profile(df_ecc=df_ecc, df_model=df_model, model='CAMS EAC4 monthly', title='Nairobi Ozone Sounding')
    except Exception as err:
        print(err)
        pass

In [ ]:
# Load ECC sonde data and extract given pressure level
from processing.ecc import SHADOZ

source = 'data/level2/ecc/shadoz'
ecc = SHADOZ()
dp = 5

# real station level
df_ecc_660 = ecc.compile_time_series_at_given_pressure(source=source, pressure_level=660, dp=dp)

# below station level
df_ecc_700 = ecc.compile_time_series_at_given_pressure(source=source, pressure_level=700, dp=dp)

# above station level
df_ecc_620 = ecc.compile_time_series_at_given_pressure(source=source, pressure_level=620, dp=dp)
df_ecc_620.head()

In [ ]:
# Load in situ O3 data from MKN, aggregate to 1h data
from processing.ebas import compile_ebas_ozone_data_into_dataframe
from processing.thermo import Thermo
from toolbox.utils import aggregate_data

source = 'data/level1'
tei49c = Thermo()
df_49c = tei49c.compile_time_series(source=source)

# remove invalid data, i.e., retain only data with flag==0
df_49c = df_49c.filter((pl.col("f_o3")=="0") | (pl.col("f_o3").is_null()))
df_49c_1h = aggregate_data(df=df_49c)

# read EBAS ozone files
data_path = 'data/wdc/ebas/air/'
df_ebas_1h = compile_ebas_ozone_data_into_dataframe(data_path=data_path)
df_ebas_1h.reset_index(inplace=True)
df_ebas_1h = pl.from_pandas(df_ebas_1h).with_columns(pl.col('dtm').cast(pl.Datetime),
                                                     pl.col('O3_0').cast(pl.Float32))

In [ ]:
# Extract dtm and ozone values from dataframes
df = df_49c_1h.select(['dtm', 'o3'])
df = df.rename({'o3': 'o3_recent_1h'})
df = pl.concat([df, df_ebas_1h.select(['dtm', 'O3_0'])], how='diagonal')
df = df.rename({'O3_0': 'o3_ebas_1h'})
df = pl.concat([df, df_ecc_660.select(['dtm', 'O3_ppbv'])], how='diagonal')
df = df.rename({'O3_ppbv': 'o3_ecc_660'})
df = pl.concat([df, df_ecc_700.select(['dtm', 'O3_ppbv'])], how='diagonal')
df = df.rename({'O3_ppbv': 'o3_ecc_700'})
df = pl.concat([df, df_ecc_620.select(['dtm', 'O3_ppbv'])], how='diagonal')
df = df.rename({'O3_ppbv': 'o3_ecc_620'})
df = df.sort('dtm')
df.describe()
df.write_parquet(file='results/ecc_ozone_nrb_vs_in_situ_ozone_mkn.parquet')

In [ ]:
# Visualize data
import polars as pl
import matplotlib.pyplot as plt
%matplotlib widget
df = pl.read_parquet(source='results/ecc_ozone_nrb_vs_in_situ_ozone_mkn.parquet')

plt.figure(figsize=(10, 6))
plt.plot(df["dtm"].to_numpy(), df["o3_ebas_1h"].to_numpy(), label="in situ", marker="o", markersize=3)
plt.plot(df["dtm"].to_numpy(), df["o3_recent_1h"].to_numpy(), label="in situ", marker="o", markersize=3)
df = df.sort(pl.col('o3_ecc_660') + pl.col('dtm'))
plt.plot(df["dtm"].to_numpy(), df["o3_ecc_660"].to_numpy(), label=f"ECC sonde at 660 mbar", linestyle="-", marker='o', markersize=3)
df = df.sort(pl.col('o3_ecc_700') + pl.col('dtm'))
plt.plot(df["dtm"].to_numpy(), df["o3_ecc_700"].to_numpy(), label=f"ECC sonde at 700 mbar", linestyle="-.", marker='o', markersize=3)
df = df.sort(pl.col('o3_ecc_620') + pl.col('dtm'))
plt.plot(df["dtm"].to_numpy(), df["o3_ecc_620"].to_numpy(), label=f"ECC sonde at 620 mbar", linestyle=":", marker='o', markersize=3)

# plt.xlabel("Date and Time")
plt.ylabel("O3 [ppbv]")
plt.title("Comparison of in situ observation at MKN and ECC sonde at NRB")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()

# Show the plot
plt.show()